## Sentiment Analysis with NLP Binary Classifier 

### Overview
This is a machine learning project on Natural Language Processing. The task given is to make binary predictions about the sentiment of the comments given in the stock_data.csv.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch

### Data Loading

In [ ]:
data = pd.read_csv('stock_data.csv')
text = data['Text'].astype("string")
sentiment = data['Sentiment']
# convert -1 in sentiment to 0
sentiment = pd.DataFrame([0 if int(label) == -1 else 1 for label in sentiment])

### Data Exploration

In [ ]:
print(text.isnull().sum())
print(sentiment.isnull().sum())
print(sentiment.value_counts())

Data is generally cleaned, but there is some class imbalance. 

### Approach 1: Using CountVectorizer and Naive Bayes

### Data Preprocessing

In [ ]:
# Split the data into training and validation and test sets
train_texts, tmp_texts, train_labels, tmp_labels = train_test_split(
    text, sentiment, test_size=0.3, random_state=42
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    tmp_texts, tmp_labels, test_size=0.6, random_state=42
)


In [ ]:
import string
import nltk
from nltk.corpus import stopwords
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
import re

In [ ]:
def preprocess_sentence(sentence):
    # remove urls
    pattern = re.compile(r'https?://\S+|www\.\S+')
    processed_text = pattern.sub(r'', sentence)
    processed_text = re.sub(r'RT[\s]+', '', processed_text)
    processed_text = re.sub(r'@\S+', '', processed_text)
    # remove non-ascii characters
    processed_text = processed_text.encode('ascii', 'ignore').decode()
    # convert to lowercase
    processed_text = nltk.word_tokenize(str(processed_text).lower())
    # remove numbers (to be assessed if this is a good idea)
    processed_text = [word for word in processed_text if not word.isdigit()]
    # remove stopwords and punctuations and the word 'user'
    stop_words = set.union(set(stopwords.words('english')), set(string.punctuation), set('user'))
    processed_text = [word for word in processed_text if word not in stop_words]
    # lemmatize
    lemmatizer = WordNetLemmatizer()
    def penn2morphy(penntag):
        """ Converts Penn Treebank tags to WordNet. """
        morphy_tag = {'NN':'n', 'JJ':'a',
                    'VB':'v', 'RB':'r'}
        try:
            return morphy_tag[penntag[:2]]
        except:
            return 'n' 
    def lemmatize_sent(text):
        # Text input is string, returns lowercased strings.
        return [lemmatizer.lemmatize(word, pos=penn2morphy(tag)) for word, tag in pos_tag(text)]
    processed_text = lemmatize_sent(processed_text) 
    return processed_text

preprocess_sentence(train_texts[0])

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vect = CountVectorizer(analyzer=preprocess_sentence)
train_X = count_vect.fit_transform(train_texts)
val_X = count_vect.transform(val_texts)
test_X = count_vect.transform(test_texts)

In [ ]:
train_X_df = pd.DataFrame(train_X.toarray(), columns=count_vect.get_feature_names_out())
features_count = train_X_df.sum(axis=0)
print("Number of words that appeared once: ", len(features_count[features_count == 1]))
print("Number of words that appeared twice: ", len(features_count[features_count == 2]))
print("Number of words that appeared 3 times: ", len(features_count[features_count == 3]))
print("Total number of words: ", len(features_count))

Seems like there are a lot of low frequency words.

In [ ]:
count_vect = CountVectorizer(analyzer=preprocess_sentence, min_df=3)
train_X = count_vect.fit_transform(train_texts)
val_X = count_vect.transform(val_texts)
test_X = count_vect.transform(test_texts)

In [ ]:
from sklearn.naive_bayes import BernoulliNB

nb_clf = BernoulliNB()
nb_clf.fit(train_X, train_labels[0])

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# predictions
predictions_valid = nb_clf.predict(val_X)
print(classification_report(val_labels[0], predictions_valid))

print("Accuracy on validation set: {:.4f}".format(accuracy_score(val_labels[0], predictions_valid)))
print("f1 score on validation set: {:.4f}".format(f1_score(val_labels[0], predictions_valid)))

# test set:
predictions_test = nb_clf.predict(test_X)
print(classification_report(test_labels[0], predictions_test))

print("Accuracy on test set: {:.4f}".format(accuracy_score(test_labels[0], predictions_test)))
print("f1 score on test set: {:.4f}".format(f1_score(test_labels[0], predictions_test)))

Some other variations were considered, for example, keeping the numbers during preprocessing and changing the minimum words frequency to be kept. However, the current model seems to be the best performing in terms of both f1 and accuracy.

### Approach 2: Using Fine-tuned DistilBert Model

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate

PRE_TRAINED_MODEL_NAME = 'distilbert-base-uncased'

# Load the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Tokenize the training and validation sets
train_encodings = tokenizer(train_texts.tolist(), add_special_tokens=True, truncation=True, padding='max_length', return_tensors='pt')
val_encodings = tokenizer(val_texts.tolist(), add_special_tokens=True, truncation=True, padding='max_length', return_tensors='pt')
test_encodings = tokenizer(test_texts.tolist(), add_special_tokens=True, truncation=True, padding='max_length', return_tensors='pt')

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


# Convert labels to tensor
train_labels_tensor = torch.tensor(train_labels.values.flatten(), dtype=torch.long)
val_labels_tensor = torch.tensor(val_labels.values.flatten(), dtype=torch.long)
test_labels_tensor = torch.tensor(test_labels.values.flatten(), dtype=torch.long)

# Create PyTorch datasets
train_dataset = SentimentDataset(train_encodings, train_labels_tensor)
val_dataset = SentimentDataset(val_encodings, val_labels_tensor)
test_dataset = SentimentDataset(test_encodings, test_labels_tensor)


f1 = evaluate.load('f1')
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return f1.compute(predictions=predictions, references=labels)

model = AutoModelForSequenceClassification.from_pretrained(PRE_TRAINED_MODEL_NAME, num_labels=2)

# to handle class imbalance
weight_for_class_0 = (1 / 3685) 
weight_for_class_1 = (1 / 2106) 
class_weights = torch.tensor([weight_for_class_0, weight_for_class_1])

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,  # Adjust as needed
    per_device_eval_batch_size=16,   # Adjust as needed
    weight_decay=0.001,
    logging_dir='./logs',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

results = trainer.evaluate()
print(results)

predictions = trainer.predict(test_dataset)
print(predictions.predictions)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

predictions = np.argmax(predictions.predictions, axis=1)
print(classification_report(test_labels_tensor, predictions))
print("Accuracy on test set: {:.4f}".format(accuracy_score(test_labels_tensor, predictions)))
print("f1 score on test set: {:.4f}".format(f1_score(test_labels_tensor, predictions)))

### Conclusion

The CountVectorizer and Naive Bayes models model has a f1 score of 0.816 while the DistilBERT model has a gf score of 0.857. DistilBERT seems to be performing slightly better than the Naive Bayes model by about 5%. However, the time taken to train the DistilBERT model was significantly larger.  

The trend is smiliar in terms of accuracy, DistilBERT still performed slightly better at 0.814

One thing to note is that the training of DistilBERT tries to maximise f1 score instead of accuracy. However, the requirement of the proect was to maximise the accuracy score, so this could possibly be higher for the DistilBERT model if it was done correctly.
